## Example 1: Earth Engine API Structure and Authentication

In [ ]:
import ee
import os
from datetime import datetime

print("Example 1: Earth Engine API Structure and Authentication")

try:
    # Explore Google Earth Engine API structure and authentication
    print("✓ Google Earth Engine API structure:")
    print(f"  Earth Engine API package: {ee}")
    
    # Show available modules and classes
    print("  API modules and components:")
    api_components = [attr for attr in dir(ee) if not attr.startswith('_')]
    for component in api_components[:10]:  # Show first 10 components
        attr_obj = getattr(ee, component)
        print(f"    - {component}: {type(attr_obj).__name__}")
    
    # Authentication and initialization patterns
    print("✓ Authentication patterns:")
    auth_methods = [
        "Service Account Authentication: Use JSON key file",
        "OAuth2 Authentication: Interactive browser auth", 
        "Application Default Credentials: Cloud environment auth",
        "API Key Authentication: For specific use cases"
    ]
    
    for method in auth_methods:
        print(f"    {method}")
    
    # Show Earth Engine data types and collections
    print("✓ Earth Engine data types:")
    ee_types = [
        "Image: Single raster image",
        "ImageCollection: Collection of images",
        "Feature: Vector data with properties", 
        "FeatureCollection: Collection of features",
        "Geometry: Geometric shapes (Point, Polygon, etc.)",
        "Reducer: Functions for data aggregation",
        "Filter: Data filtering operations"
    ]
    
    for ee_type in ee_types:
        print(f"    - {ee_type}")
    
    # Common API endpoints and services
    print("✓ API services:")
    services = [
        "Data Catalog: Browse available datasets",
        "Processing Service: Run Earth Engine algorithms",
        "Export Service: Export results to Drive/Cloud Storage",
        "Asset Service: Manage user assets",
        "Task Service: Monitor processing tasks"
    ]
    
    for service in services:
        print(f"    ✓ {service}")
    
    # Authentication requirements
    print("✓ Authentication requirements:")
    requirements = [
        "1. Google Cloud Project with Earth Engine API enabled",
        "2. Service account or OAuth2 credentials",
        "3. Earth Engine Python API installed",
        "4. Valid authentication token"
    ]
    
    for req in requirements:
        print(f"    {req}")
    
    print("✓ Earth Engine API structure examined successfully")
    print("  Note: Actual usage requires valid Google Cloud credentials and ee.Initialize()")

except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: Install with 'pip install earthengine-api' to use Google Earth Engine")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"earthengine_api example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 finished")

## Example 2: Data Catalog and Collection Access Patterns

In [ ]:
import ee
from ee import apitestcase  # earthengine-api's own offline API fake (shipped in the package)
from datetime import datetime, timedelta

print("Example 2: Data Catalog and Collection Access Patterns")

try:
    # Demonstrate Earth Engine data catalog and collection access patterns
    print("✓ Earth Engine data catalog access:")
    
    # Show Earth Engine module structure
    print("  Earth Engine API components:")
    ee_components = [
        ("ee.Image", "Single raster image"),
        ("ee.ImageCollection", "Collection of images"),
        ("ee.Feature", "Vector data with properties"),
        ("ee.FeatureCollection", "Collection of features"),
        ("ee.Geometry", "Geometric shapes"),
        ("ee.Filter", "Data filtering operations"),
        ("ee.Reducer", "Data aggregation functions")
    ]
    
    for component, description in ee_components:
        print(f"    {component}: {description}")
    
    # Demonstrate collection access patterns (without authentication)
    print("✓ Collection access patterns:")
    
    # Show how to reference collections
    collection_examples = [
        "LANDSAT/LC08/C02/T1_L2",
        "COPERNICUS/S2_SR", 
        "MODIS/061/MOD13Q1",
        "NASA/GLDAS/V021/NOAH/G025/T3H"
    ]
    
    # Collections are lazy client-side references, but building one still needs an initialized
    # client (without it: EEException "not initialized"). `ee.apitestcase` initializes OFFLINE: it
    # serves the algorithm catalogue from the package's bundled tests/algorithms.json and stubs the
    # cloud resource, so no request and no credentials. tearDown() + ee.Reset() restore the
    # uninitialized state, so later cells see the library exactly as before.
    offline_session = apitestcase.ApiTestCase()
    offline_session.setUp()
    try:
        for collection_id in collection_examples:
            print(f"  {collection_id}")
            collection = ee.ImageCollection(collection_id)
            assert isinstance(collection, ee.ImageCollection), type(collection)
            assert collection_id in collection.serialize(), collection.serialize()
            print(f"    ✓ Collection reference created: {type(collection)}")
        # A filter chain is built client-side too; nothing is evaluated.
        chain = (ee.ImageCollection(collection_examples[0])
                 .filterDate('2020-01-01', '2020-12-31')
                 .filter(ee.Filter.lt('CLOUD_COVER', 20)))
        graph = chain.serialize()
        for name in ("ImageCollection.load", "Collection.filter", "Filter.dateRangeContains", "Filter.lessThan"):
            assert f'"{name}"' in graph, (name, graph[:400])
        print("    ✓ filterDate + filter chain serialized without a request")
    finally:
        offline_session.tearDown()
        ee.Reset()
    
    # Show Earth Engine API pattern examples
    print("✓ Earth Engine API patterns:")
    api_examples = [
        "ee.ImageCollection('COLLECTION_ID')",
        "collection.filterDate('2020-01-01', '2020-12-31')",
        "collection.filterBounds(geometry)",
        "collection.filter(ee.Filter.lt('CLOUD_COVER', 20))",
        "collection.select(['B4', 'B3', 'B2'])",
        "collection.map(lambda image: image.normalizedDifference(['B5', 'B4']))",
        "collection.reduce(ee.Reducer.median())",
        "ee.Export.image.toDrive(image, 'description')"
    ]
    
    for example in api_examples:
        print(f"    {example}")
    
    # Show filter types
    print("✓ Earth Engine filter types:")
    filter_types = [
        ("ee.Filter.date", "Temporal filtering"),
        ("ee.Filter.bounds", "Spatial filtering"),  
        ("ee.Filter.eq", "Equality filtering"),
        ("ee.Filter.lt", "Less than filtering"),
        ("ee.Filter.gt", "Greater than filtering"),
        ("ee.Filter.listContains", "List membership"),
        ("ee.Filter.dayOfYear", "Day of year filtering"),
        ("ee.Filter.calendarRange", "Calendar range filtering")
    ]
    
    for filter_func, description in filter_types:
        print(f"    {filter_func}: {description}")
    
    # Data access workflow
    print("✓ Data access workflow:")
    workflow_steps = [
        "1. Authenticate: ee.Authenticate() (one-time setup)",
        "2. Initialize: ee.Initialize() (per session)",
        "3. Load collection: ee.ImageCollection('COLLECTION_ID')",
        "4. Apply filters: temporal, spatial, metadata",
        "5. Select bands: collection.select(['band1', 'band2'])",
        "6. Process data: map functions, reduce operations",
        "7. Export results: ee.Export.image/table.toDrive/toAsset"
    ]
    
    for step in workflow_steps:
        print(f"    {step}")
        
    # Show geometry creation
    print("✓ Geometry creation examples:")
    geometry_examples = [
        "ee.Geometry.Point([lon, lat])",
        "ee.Geometry.Rectangle([west, south, east, north])",
        "ee.Geometry.Polygon([[[lon1, lat1], [lon2, lat2], ...]])",
        "ee.Geometry.LineString([[lon1, lat1], [lon2, lat2]])"
    ]
    
    for example in geometry_examples:
        print(f"    {example}")

    print("✓ Data catalog and collection access demonstration completed")

except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Install with: pip install earthengine-api")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    print("  Note: Most Earth Engine operations require authentication")
    raise AssertionError(
        f"earthengine_api example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 finished")

## Example 3: Image Processing and Analysis Operations

In [ ]:
import ee
import numpy as np
from datetime import datetime

print("Example 3: Image Processing and Analysis Operations")

try:
    # Demonstrate Earth Engine image processing and analysis operations
    print("✓ Earth Engine image processing capabilities:")
    
    # Show available image processing methods
    image_methods = [
        ("normalizedDifference", "Calculate normalized difference indices"),
        ("expression", "Apply mathematical expressions"),
        ("select", "Select specific bands"),
        ("rename", "Rename bands"),
        ("addBands", "Add additional bands"),
        ("mask", "Apply masking operations"),
        ("clip", "Clip to geometry"),
        ("reproject", "Change projection"),
        ("resample", "Change resolution"),
        ("convolve", "Apply convolution filters")
    ]
    
    for method, description in image_methods:
        print(f"    ee.Image.{method}(): {description}")
    
    # Show spectral index calculations using ee
    print("✓ Spectral index calculations:")
    
    try:
        # Create sample image reference (won't execute without auth)
        print("  NDVI calculation pattern:")
        print("    ndvi = image.normalizedDifference(['B5', 'B4']).rename('NDVI')")
        
        print("  NDWI calculation pattern:")
        print("    ndwi = image.normalizedDifference(['B3', 'B5']).rename('NDWI')")
        
        print("  EVI calculation pattern:")
        print("    evi = image.expression('2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {")
        print("      'NIR': image.select('B5'),")
        print("      'RED': image.select('B4'),") 
        print("      'BLUE': image.select('B2')")
        print("    }).rename('EVI')")
        
    except Exception as e:
        print(f"    Note: {str(e)[:50]}...")
        raise
    
    # Show filtering operations
    print("✓ Image filtering operations:")
    filter_methods = [
        ("focal_median", "Median filter for noise reduction"),
        ("focal_mean", "Mean smoothing filter"),
        ("focal_max", "Maximum filter"),
        ("focal_min", "Minimum filter"),
        ("convolve", "Custom convolution kernels"),
        ("gaussianBlur", "Gaussian blur"),
        ("gradient", "Gradient computation")
    ]
    
    for method, description in filter_methods:
        print(f"    ee.Image.{method}(): {description}")
    
    # Show kernel creation
    print("✓ Kernel creation for filtering:")
    kernel_examples = [
        "ee.Kernel.square(radius=1)",
        "ee.Kernel.circle(radius=2)", 
        "ee.Kernel.gaussian(radius=3, sigma=1)",
        "ee.Kernel.laplacian8()",
        "ee.Kernel.sobel()",
        "ee.Kernel.roberts()"
    ]
    
    for example in kernel_examples:
        print(f"    {example}")
    
    # Show classification approaches
    print("✓ Classification methods:")
    classifiers = [
        ("ee.Classifier.smileCart", "Classification and Regression Trees"),
        ("ee.Classifier.smileRandomForest", "Random Forest classifier"),
        ("ee.Classifier.libsvm", "Support Vector Machine"),
        ("ee.Classifier.smileNaiveBayes", "Naive Bayes classifier"),
        ("ee.Classifier.smileGradientTreeBoost", "Gradient Boosted Trees"),
        ("ee.Classifier.minimumDistance", "Minimum Distance classifier")
    ]
    
    for classifier, description in classifiers:
        print(f"    {classifier}(): {description}")
    
    # Show reducer operations
    print("✓ Reduction operations:")
    reducers = [
        ("ee.Reducer.mean", "Calculate mean values"),
        ("ee.Reducer.median", "Calculate median values"),
        ("ee.Reducer.max", "Find maximum values"),
        ("ee.Reducer.min", "Find minimum values"),
        ("ee.Reducer.stdDev", "Calculate standard deviation"),
        ("ee.Reducer.percentile", "Calculate percentiles"),
        ("ee.Reducer.count", "Count non-null values"),
        ("ee.Reducer.sum", "Sum values")
    ]
    
    for reducer, description in reducers:
        print(f"    {reducer}(): {description}")
    
    # Processing workflow example using ee
    print("✓ Image processing workflow:")
    workflow = [
        "1. Load image: ee.Image('IMAGE_ID')",
        "2. Select bands: image.select(['B4', 'B3', 'B2'])",
        "3. Calculate indices: image.normalizedDifference(['B5', 'B4'])",
        "4. Apply filters: image.focal_median(ee.Kernel.circle(1))",
        "5. Mask clouds: image.updateMask(cloud_mask)",
        "6. Classify: image.classify(classifier)",
        "7. Reduce regions: image.reduceRegions(collection, reducer)"
    ]
    
    for step in workflow:
        print(f"    {step}")
    
    # Analysis capabilities using Earth Engine
    print("✓ Earth Engine analysis capabilities:")
    capabilities = [
        "Time series analysis with ee.ImageCollection",
        "Change detection using image differencing",
        "Land cover classification with ee.Classifier",
        "Vegetation monitoring using spectral indices",
        "Water detection with NDWI and thresholding",
        "Urban analysis using built-up indices",
        "Agricultural monitoring with crop masks",
        "Disaster assessment using before/after imagery"
    ]
    
    for capability in capabilities:
        print(f"    ✓ {capability}")

    print("✓ Image processing and analysis demonstration completed")

except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Install with: pip install earthengine-api")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    print("  Note: Most operations require ee.Initialize() with authentication")
    raise AssertionError(
        f"earthengine_api example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 finished")

## Example 4: Export and Task Management

In [ ]:
import ee
import time
from datetime import datetime

print("Example 4: Export and Task Management")

try:
    # Demonstrate Earth Engine export and task management
    print("✓ Earth Engine export and task management:")
    
    # Show export destinations using ee
    print("  Export destinations:")
    export_destinations = [
        ("ee.Export.image.toDrive", "Export to Google Drive"),
        ("ee.Export.image.toCloudStorage", "Export to Google Cloud Storage"),
        ("ee.Export.image.toAsset", "Export to Earth Engine Asset"),
        ("ee.Export.table.toDrive", "Export table to Google Drive"),
        ("ee.Export.table.toCloudStorage", "Export table to Cloud Storage"),
        ("ee.Export.table.toBigQuery", "Export table to BigQuery"),
        ("ee.Export.video.toDrive", "Export video to Google Drive"),
        ("ee.Export.video.toCloudStorage", "Export video to Cloud Storage")
    ]
    
    for export_func, description in export_destinations:
        print(f"    {export_func}(): {description}")
    
    # Show export format options
    print("✓ Export format options:")
    formats = [
        ("Image exports", ["GeoTIFF", "TFRecord", "JPEG", "PNG"]),
        ("Table exports", ["CSV", "SHP", "KML", "GeoJSON", "TFRecord"]),
        ("Video exports", ["MP4", "GIF"])
    ]
    
    for data_type, format_list in formats:
        print(f"    {data_type}: {', '.join(format_list)}")
    
    # Show task management functions
    print("✓ Task management functions:")
    task_functions = [
        ("ee.batch.Task.list", "List all tasks"),
        ("task.start", "Start a task"),
        ("task.status", "Get task status"),
        ("task.cancel", "Cancel a running task"),
        ("ee.batch.Task.active", "Get active tasks"),
        ("ee.batch.Task.ready", "Get ready tasks"),
        ("ee.batch.Task.completed", "Get completed tasks"),
        ("ee.batch.Task.failed", "Get failed tasks")
    ]
    
    for func, description in task_functions:
        print(f"    {func}(): {description}")
    
    # Show task states
    print("✓ Task states in Earth Engine:")
    task_states = [
        ("READY", "Task created but not started"),
        ("RUNNING", "Task is currently executing"),
        ("COMPLETED", "Task finished successfully"),
        ("FAILED", "Task failed with error"),
        ("CANCELLED", "Task was cancelled by user"),
        ("CANCEL_REQUESTED", "Cancellation requested")
    ]
    
    for state, description in task_states:
        print(f"    {state}: {description}")
    
    # Export configuration examples
    print("✓ Export configuration examples:")
    
    print("  Image export to Drive:")
    drive_config = [
        "ee.Export.image.toDrive({",
        "  'image': processed_image,",
        "  'description': 'landsat_composite',",
        "  'scale': 30,",
        "  'region': study_area,",
        "  'folder': 'EarthEngine',",
        "  'fileFormat': 'GeoTIFF'",
        "})"
    ]
    for line in drive_config:
        print(f"    {line}")
    
    print("  Table export to BigQuery:")
    bq_config = [
        "ee.Export.table.toBigQuery({",
        "  'collection': feature_collection,",
        "  'description': 'field_samples',",
        "  'table': 'project.dataset.table',",
        "  'fileFormat': 'CSV'",
        "})"
    ]
    for line in bq_config:
        print(f"    {line}")
    
    # Show monitoring patterns
    print("✓ Task monitoring patterns:")
    monitoring_examples = [
        "# Check all tasks",
        "tasks = ee.batch.Task.list()",
        "for task in tasks[:5]:",
        "    print(f'{task.id}: {task.state}')",
        "",
        "# Monitor specific task",
        "task = export_task",
        "while task.active():",
        "    print(f'Status: {task.status()}')",
        "    time.sleep(30)"
    ]
    
    for line in monitoring_examples:
        print(f"    {line}")
    
    # Best practices
    print("✓ Export best practices:")
    best_practices = [
        "Use appropriate scale for your analysis needs",
        "Set reasonable region boundaries to avoid timeouts",
        "Use descriptive names for exported assets",
        "Monitor export progress regularly",
        "Handle failed exports with retry logic",
        "Clean up completed tasks periodically",
        "Use batch exports for multiple similar tasks",
        "Consider Earth Engine quotas and limits"
    ]
    
    for practice in best_practices:
        print(f"    ✓ {practice}")
    
    # Common export parameters
    print("✓ Common export parameters:")
    parameters = [
        ("scale", "Output resolution in meters"),
        ("region", "Geographic bounds for export"),
        ("crs", "Coordinate reference system"),
        ("crsTransform", "Affine transformation matrix"),
        ("maxPixels", "Maximum number of pixels"),
        ("shardSize", "Size of export shards"),
        ("fileDimensions", "Dimensions for tiled exports"),
        ("skipEmptyTiles", "Skip tiles with no data")
    ]
    
    for param, description in parameters:
        print(f"    {param}: {description}")

    print("✓ Export and task management demonstration completed")

except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Install with: pip install earthengine-api")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    print("  Note: Export operations require authentication and valid projects")
    raise AssertionError(
        f"earthengine_api example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 finished")

## Example 5: Production Integration and Best Practices

In [ ]:
import ee
import logging
from datetime import datetime
from typing import Optional, Dict, List, Any

print("Example 5: Production Integration and Best Practices")

try:
    # Demonstrate production integration and best practices for Earth Engine API
    print("✓ Earth Engine production integration and best practices:")
    
    # Show authentication methods
    print("  Authentication methods:")
    auth_methods = [
        ("ee.Authenticate()", "Interactive browser authentication"),
        ("ee.ServiceAccountCredentials()", "Service account authentication"),
        ("ee.Initialize()", "Initialize with default credentials"),
        ("ee.Initialize(credentials)", "Initialize with specific credentials"),
        ("ee.Initialize(opt_url)", "Initialize with custom server URL")
    ]
    
    for method, description in auth_methods:
        print(f"    {method}: {description}")
    
    # Show initialization patterns
    print("✓ Initialization patterns:")
    init_patterns = [
        "# Browser authentication (development)",
        "ee.Authenticate()",
        "ee.Initialize()",
        "",
        "# Service account (production)",
        "credentials = ee.ServiceAccountCredentials(email, key_file)",
        "ee.Initialize(credentials)",
        "",
        "# High-volume endpoint",
        "ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')"
    ]
    
    for line in init_patterns:
        print(f"    {line}")
    
    # Show error handling patterns
    print("✓ Error handling with Earth Engine:")
    error_types = [
        ("ee.EEException", "General Earth Engine errors"),
        ("HttpError", "HTTP communication errors"),
        ("AuthenticationError", "Authentication failures"),
        ("QuotaExceededError", "Usage quota exceeded"),
        ("TimeoutError", "Request timeout errors"),
        ("ServerError", "Server-side processing errors")
    ]
    
    for error_type, description in error_types:
        print(f"    {error_type}: {description}")
    
    # Show retry patterns
    print("✓ Retry patterns for robustness:")
    retry_example = [
        "import time",
        "from google.api_core import retry",
        "",
        "def robust_ee_operation(operation, max_retries=3):",
        "    for attempt in range(max_retries):",
        "        try:",
        "            return operation()",
        "        except ee.EEException as e:",
        "            if attempt == max_retries - 1:",
        "                raise",
        "            wait_time = 2 ** attempt",
        "            time.sleep(wait_time)"
    ]
    
    for line in retry_example:
        print(f"    {line}")
    
    # Show quota management
    print("✓ Quota management:")
    quota_limits = [
        ("Concurrent requests", "1000 per user"),
        ("Earth Engine Units (EEUs)", "Daily and per-request limits"),
        ("Export size", "10GB per export"),
        ("Asset storage", "250GB per user"),
        ("Compute time", "5 minutes per request"),
        ("Memory usage", "Limited by server capacity")
    ]
    
    for limit_type, limit_value in quota_limits:
        print(f"    {limit_type}: {limit_value}")
    
    # Show optimization strategies
    print("✓ Performance optimization strategies:")
    optimizations = [
        "Use .select() to reduce data transfer",
        "Apply filters early in processing chain",
        "Leverage server-side operations vs .getInfo()",
        "Use .limit() for testing with smaller datasets",
        "Cache intermediate results appropriately",
        "Optimize spatial and temporal scales",
        "Use pyramiding for multi-scale analysis",
        "Batch similar operations together"
    ]
    
    for optimization in optimizations:
        print(f"    ✓ {optimization}")
    
    # Show monitoring and logging
    print("✓ Monitoring and logging setup:")
    monitoring_setup = [
        "import logging",
        "",
        "# Setup logging",
        "logging.basicConfig(",
        "    level=logging.INFO,",
        "    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'",
        ")",
        "logger = logging.getLogger('earthengine')",
        "",
        "# Log Earth Engine operations",
        "logger.info('Starting Earth Engine processing')",
        "try:",
        "    result = ee_operation()",
        "    logger.info(f'Operation completed: {result}')",
        "except Exception as e:",
        "    logger.error(f'Operation failed: {e}')"
    ]
    
    for line in monitoring_setup:
        print(f"    {line}")
    
    # Show production workflow
    print("✓ Production workflow example:")
    workflow = [
        "1. Initialize Earth Engine with service account",
        "2. Validate input parameters and quotas",
        "3. Apply defensive programming patterns",
        "4. Implement comprehensive error handling",
        "5. Use retry logic for transient failures",
        "6. Monitor processing progress and logs",
        "7. Handle export tasks asynchronously",
        "8. Implement proper cleanup procedures"
    ]
    
    for step in workflow:
        print(f"    {step}")
    
    # Show deployment considerations
    print("✓ Deployment considerations:")
    deployment_tips = [
        "Store service account keys securely",
        "Use environment variables for configuration",
        "Implement proper access controls",
        "Set up monitoring and alerting",
        "Plan for quota management",
        "Document API usage patterns",
        "Test with representative workloads",
        "Implement graceful degradation"
    ]
    
    for tip in deployment_tips:
        print(f"    • {tip}")
    
    # Show common patterns for production use
    print("✓ Common production patterns:")
    patterns = [
        ("Batch processing", "Process multiple images/regions together"),
        ("Async workflows", "Use task queues for long operations"),
        ("Result caching", "Store intermediate results as assets"),
        ("Error recovery", "Implement checkpointing and resume"),
        ("Load balancing", "Distribute work across time/resources"),
        ("Version control", "Track code and data versions"),
        ("Testing", "Unit and integration tests for pipelines"),
        ("Documentation", "Maintain API usage documentation")
    ]
    
    for pattern, description in patterns:
        print(f"    {pattern}: {description}")

    print("✓ Production integration and best practices demonstration completed")

except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Install with: pip install earthengine-api")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    print("  Note: Production deployment requires proper authentication setup")
    raise AssertionError(
        f"earthengine_api example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 finished")